In [4]:

from pathlib import Path

import fastf1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    log_loss,
)
from xgboost import XGBClassifier

# Enable FastF1 cache
CACHE_DIR = Path("data/fastf1_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

fastf1.Cache.enable_cache(str(CACHE_DIR))

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("FastF1 cache enabled:", CACHE_DIR.resolve())
print("FastF1 version:", fastf1.__version__)


FastF1 cache enabled: C:\Tanay\development and dsa\F1-Race-Analysis\data\fastf1_cache
FastF1 version: 3.8.3


In [5]:

# Example: 2025 British Grand Prix qualifying
session = fastf1.get_session(2025, "British Grand Prix", "Q")
session.load()

qualifying_results = session.results

display(
    qualifying_results[
        ["Abbreviation", "TeamName", "Position"]
    ].sort_values("Position")
)


core           INFO 	Loading data for British Grand Prix - Qualifying [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 

,Abbreviation,TeamName,Position
1,VER,Red Bull Racing,1.000
81,PIA,McLaren,2.000
4,NOR,McLaren,3.000
63,RUS,Mercedes,4.000
44,HAM,Ferrari,5.000
16,LEC,Ferrari,6.000
12,ANT,Mercedes,7.000
87,BEA,Haas F1 Team,8.000
14,ALO,Aston Martin,9.000
10,GAS,Alpine,10.000


In [6]:

import fastf1
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

SEASONS = range(2022, 2026)

def collect_season_data(year):
    """Collect qualifying and race results for one F1 season."""
    schedule = fastf1.get_event_schedule(year, include_testing=False)

    season_rows = []

    for _, event in schedule.iterrows():
        event_name = event["EventName"]
        round_number = event["RoundNumber"]

        if pd.isna(round_number) or round_number == 0:
            continue

        try:
            print(f"Processing {year}: {event_name}")

            # Load qualifying
            quali = fastf1.get_session(year, event_name, "Q")
            quali.load(laps=False, telemetry=False, weather=False)

            # Load Grand Prix race
            race = fastf1.get_session(year, event_name, "R")
            race.load(laps=False, telemetry=False, weather=False)

            q = quali.results.copy()
            r = race.results.copy()

            # Qualifying features
            q = q[[
                "Abbreviation",
                "TeamName",
                "Position"
            ]].rename(columns={
                "Position": "QualifyingPosition"
            })

            # Race target and starting grid
            r = r[[
                "Abbreviation",
                "GridPosition",
                "Position",
                "Status"
            ]].rename(columns={
                "Position": "FinishPosition"
            })

            # Join qualifying and race results by driver
            merged = q.merge(
                r,
                on="Abbreviation",
                how="inner",
                suffixes=("_Quali", "_Race")
            )

            merged["Season"] = year
            merged["Round"] = int(round_number)
            merged["EventName"] = event_name

            # One means the driver won the race
            merged["Winner"] = (
                pd.to_numeric(
                    merged["FinishPosition"], errors="coerce"
                ) == 1
            ).astype(int)

            season_rows.append(merged)

        except Exception as e:
            print(f"Skipped {year} {event_name}: {e}")

    if not season_rows:
        return pd.DataFrame()

    return pd.concat(season_rows, ignore_index=True)


In [7]:

all_seasons = []

for year in SEASONS:
    season_df = collect_season_data(year)

    if not season_df.empty:
        all_seasons.append(season_df)

if not all_seasons:
    raise RuntimeError(
        "No data collected. Check FastF1 access and session errors."
    )

f1_data = pd.concat(all_seasons, ignore_index=True)

print("Dataset shape:", f1_data.shape)
display(f1_data.head())


core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


Processing 2022: Bahrain Grand Prix


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for race_control_messages. Loading data...
_api           INFO 	Fetching race control messages...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['16', '1', '55', '11', '44', '77', '20', '14', '63', '10', '31', '47', '4', '23', '24', '22', '27', '3', '18', '6']
core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has be

Processing 2022: Saudi Arabian Grand Prix


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for race_control_messages. Loading data...
_api           INFO 	Fetching race control messages...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['11', '16', '55', '1', '31', '63', '14', '77', '10', '20', '4', '3', '24', '47', '18', '44', '23', '27', '6', '22']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data 

...

Processing 2022: Hungarian Grand Prix


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for race_control_messages. Loading data...
_api           INFO 	Fetching race control messages...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['63', '55', '16', '4', '31', '14', '44', '77', '3', '1', '11', '24', '20', '18', '47', '22', '23', '5', '10', '6']
core           INFO 	Loading data for Hungarian Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has b

Processing 2022: Belgian Grand Prix


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for race_control_messages. Loading data...
_api           INFO 	Fetching race control messages...


KeyboardInterrupt: 

In [ ]:

output_path = DATA_DIR / "f1_race_results_2022_2025.csv"

f1_data.to_csv(output_path, index=False)

print(f"Saved dataset to: {output_path.resolve()}")
print("Unique races:", f1_data[["Season", "Round"]].drop_duplicates().shape[0])
print("Total driver-race entries:", len(f1_data))
